## I. Importing libraries

In [1]:
import sys
import numpy as np
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import random
import pandas as pd

In [2]:
import utils, model

## II. Preprocessing data

In [4]:
train_data = np.load("encoded_data/train_data_kmer.npz")
train_features, train_labels = train_data["features"], train_data["labels"]

test_data = np.load("encoded_data/test_data_kmer.npz")
test_features, test_labels = test_data["features"], test_data["labels"]

## III. Training model

### 1. Define seed

In [5]:
# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # If using GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### 2. Define data

In [6]:
train_features, val_features, train_labels, val_labels = train_test_split(train_features, train_labels, test_size=0.2, stratify=train_labels, random_state=42)

# Convert NumPy arrays to PyTorch tensors
train_features = torch.from_numpy(train_features).float()
train_labels = torch.from_numpy(train_labels).float()
val_features = torch.from_numpy(val_features).float()
val_labels = torch.from_numpy(val_labels).float()
test_features = torch.from_numpy(test_features).float()
test_labels = torch.from_numpy(test_labels).float()

In [7]:
# Create subsets based on the indices
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

In [8]:
# Optionally, create DataLoader objects for batch processing during training
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 3. Define model and training

In [ ]:
set_seed(42)
train_losses = []
val_losses = []

# Initialize Model
input_length = train_features.shape[1]
cnn_model = model.CNNWithAttention(input_length)

# Loss Function & Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model.to(device)

# Track Best Model
best_val_loss = float('inf')
best_metrics = []

num_epochs = 5

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    # Train
    train_loss = utils.train(cnn_model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)

    # Validate
    print("\n------ VALIDATION ------")
    val_loss, val_labels, val_probs = utils.get_predictions(cnn_model, val_loader, criterion, device)
    val_metrics = utils.evaluate(val_labels, val_probs)
    val_losses.append(val_loss)

    # Test
    print("\n------ TEST ------")
    test_loss, test_labels, test_probs = utils.get_predictions(cnn_model, test_loader, criterion, device)
    test_metrics = utils.evaluate(test_labels, test_probs)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_metrics = [
            {'Metric Type': 'Validation', 'Epoch': epoch + 1, 'Loss': val_loss, **val_metrics},
            {'Metric Type': 'Test', 'Epoch': epoch + 1, 'Loss': test_loss, **test_metrics}
        ]

        print(f'✅ Best model saved with validation loss: {best_val_loss:.4f}')

    print('=' * 80)

Epoch 1/5

------ VALIDATION ------
Loss: 0.6935
Accuracy: 0.5000
ROC AUC: 0.2332
Specificity: 1.0000
Sensitivity: 0.0000
Precision: 0.0000
F1 Score: 0.0000
Balanced Accuracy: 0.5000
MCC: 0.0000

------ TEST ------


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Loss: 0.6931
Accuracy: 0.5000
ROC AUC: 0.2178
Specificity: 1.0000
Sensitivity: 0.0000
Precision: 0.0000
F1 Score: 0.0000
Balanced Accuracy: 0.5000
MCC: 0.0000
✅ Best model saved with validation loss: 0.6935
Epoch 2/5


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



------ VALIDATION ------
Loss: 0.6932
Accuracy: 0.5000
ROC AUC: 0.5213
Specificity: 1.0000
Sensitivity: 0.0000
Precision: 0.0000
F1 Score: 0.0000
Balanced Accuracy: 0.5000
MCC: 0.0000

------ TEST ------


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Loss: 0.6931
Accuracy: 0.5000
ROC AUC: 0.4889
Specificity: 1.0000
Sensitivity: 0.0000
Precision: 0.0000
F1 Score: 0.0000
Balanced Accuracy: 0.5000
MCC: 0.0000
✅ Best model saved with validation loss: 0.6932
Epoch 3/5


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



------ VALIDATION ------
Loss: 0.6932
Accuracy: 0.5000
ROC AUC: 0.6420
Specificity: 0.0000
Sensitivity: 1.0000
Precision: 0.5000
F1 Score: 0.6667
Balanced Accuracy: 0.5000
MCC: 0.0000

------ TEST ------
Loss: 0.6933
Accuracy: 0.5000
ROC AUC: 0.6216
Specificity: 0.0000
Sensitivity: 1.0000
Precision: 0.5000
F1 Score: 0.6667
Balanced Accuracy: 0.5000
MCC: 0.0000
Epoch 4/5
